In [1]:
# Célula 1 (revisada) — DuckDB + MinIO com Secrets Manager
import duckdb

con = duckdb.connect("../data/curso_analytics.duckdb")
con.execute("INSTALL httpfs; LOAD httpfs;")

# Segredo TEMPORÁRIO: vale para esta conexão quando feita entre containers Docker. Rode esta célula sempre que reabrir o notebook.
con.execute("""
    CREATE OR REPLACE SECRET minio (
        TYPE s3,
        KEY_ID 'minioadmin',
        SECRET 'minioadmin123',
        ENDPOINT 'host.docker.internal:9000',
        URL_STYLE 'path',
        USE_SSL false,
        REGION 'us-east-1'
    );
""")

# Segredo Temporário usando localhost: vale para as conexões feitas pelo host. Rode esta célula apenas uma vez.
#con.execute("""
#    CREATE OR REPLACE SECRET minio (
#        TYPE s3,
#        KEY_ID 'minioadmin',
#        SECRET 'minioadmin123',
#        ENDPOINT 'localhost:9000',
#        URL_STYLE 'path',
#        USE_SSL false,
#        REGION 'us-east-1'
#    );
#""")

print(con.sql("SELECT name, type, persistent FROM duckdb_secrets()"))

┌─────────┬─────────┬────────────┐
│  name   │  type   │ persistent │
│ varchar │ varchar │  boolean   │
├─────────┼─────────┼────────────┤
│ minio   │ s3      │ false      │
└─────────┴─────────┴────────────┘



In [3]:
# Célula 2 (revisada) — Catálogo Iceberg (SQLite local + MinIO)
import io, os, time
from datetime import datetime, timezone

import boto3
import pandas as pd
import pyarrow as pa
from pyiceberg.catalog.sql import SqlCatalog

os.makedirs("../data/iceberg_catalog", exist_ok=True)

MINIO = {
    "s3.endpoint": "http://host.docker.internal:9000",
    "s3.access-key-id": "minioadmin",
    "s3.secret-access-key": "minioadmin123",
    "s3.region": "us-east-1",            # evita a tentativa de descobrir a região do bucket
}

#MINIO = {
#    "s3.endpoint": "http://localhost:9000",
#    "s3.access-key-id": "minioadmin",
#    "s3.secret-access-key": "minioadmin123",
#    "s3.region": "us-east-1",            # evita a tentativa de descobrir a região do bucket
#}


catalog = SqlCatalog(
    "curso_lakehouse",
    **{"uri": "sqlite:///../data/iceberg_catalog/catalog.db", "warehouse": "s3://silver", **MINIO},
)
catalog.create_namespace_if_not_exists("bronze_processado")
print("Namespaces:", catalog.list_namespaces())


Namespaces: [('bronze_processado',)]


In [4]:
# Célula 3 — Schema fortemente tipado + particionamento oculto (hidden partitioning)
from pyiceberg.schema import Schema
from pyiceberg.types import (NestedField, LongType, IntegerType, StringType,
                             DoubleType, TimestampType, BooleanType)
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import MonthTransform

schema_pedidos = Schema(
    NestedField(1,  "id_pedido",         LongType(),      required=True),
    NestedField(2,  "id_cliente",        LongType(),      required=True),
    NestedField(3,  "id_produto",        LongType(),      required=True),
    NestedField(4,  "categoria",         StringType()),
    NestedField(5,  "valor_unitario",    DoubleType()),
    NestedField(6,  "quantidade",        IntegerType()),
    NestedField(7,  "valor_total",       DoubleType()),
    NestedField(8,  "status_pedido",     StringType()),
    NestedField(9,  "regiao",            StringType()),
    NestedField(10, "data_pedido",       TimestampType()),
    NestedField(11, "avaliacao_cliente", IntegerType()),
    NestedField(12, "_fonte",            StringType()),
    NestedField(13, "_ingerido_em",      StringType()),
    identifier_field_ids=[1],            # chave lógica: usada pelo upsert
)

# O particionamento é declarado sobre uma TRANSFORMAÇÃO da coluna (mês de data_pedido).
# Quem consulta filtra por data_pedido e não precisa conhecer a estrutura de pastas.
spec_particao = PartitionSpec(
    PartitionField(source_id=10, field_id=1000, transform=MonthTransform(), name="data_pedido_mes")
)

NOME_TABELA = "bronze_processado.pedidos_ecommerce"
tabela = catalog.create_table_if_not_exists(
    identifier=NOME_TABELA,
    schema=schema_pedidos,
    partition_spec=spec_particao,
    properties={
        "write.parquet.compression-codec": "zstd",
        "write.metadata.compression-codec": "gzip",
    },
)
print(tabela.schema())
print(tabela.spec())
print("Localização:", tabela.location())

table {
  1: id_pedido: required long
  2: id_cliente: required long
  3: id_produto: required long
  4: categoria: optional string
  5: valor_unitario: optional double
  6: quantidade: optional int
  7: valor_total: optional double
  8: status_pedido: optional string
  9: regiao: optional string
  10: data_pedido: optional timestamp
  11: avaliacao_cliente: optional int
  12: _fonte: optional string
  13: _ingerido_em: optional string
}
[
  1000: data_pedido_mes: month(10)
]
Localização: s3://silver/bronze_processado/pedidos_ecommerce
